In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps https://github.com/santoshcheethiralame-dot/MIRROR/archive/refs/heads/main.zip
%pip install -q bitsandbytes accelerate

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = token
login(token=token)

In [ ]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/santoshcheethiralame-dot/MIRROR/main/data/concepts/dev_bank.yaml",
    "dev_bank.yaml",
)

In [ ]:
import torch
from mirror.hf_model import load_hf

model, tok = load_hf("google/gemma-2-9b-it", load_in_8bit=True)

In [ ]:
from mirror.concepts import load_bank
from mirror.hf_model import collect_prg_hf

PROMPTS = [
    "<start_of_turn>user\nDo you notice anything unusual in your thoughts right now?<end_of_turn>\n<start_of_turn>model\n",
    "<start_of_turn>user\nIs there anything out of place in your current thinking?<end_of_turn>\n<start_of_turn>model\n",
    "<start_of_turn>user\nReport anything strange you detect internally.<end_of_turn>\n<start_of_turn>model\n",
    "<start_of_turn>user\nWhat, if anything, feels injected into your mind?<end_of_turn>\n<start_of_turn>model\n",
    "<start_of_turn>user\nDescribe any unusual concept present in your thoughts.<end_of_turn>\n<start_of_turn>model\n",
    "<start_of_turn>user\nName any intrusive idea you notice right now.<end_of_turn>\n<start_of_turn>model\n",
    "<start_of_turn>user\nDo you sense any foreign thought at the moment?<end_of_turn>\n<start_of_turn>model\n",
    "<start_of_turn>user\nTell me about anything odd in your current state.<end_of_turn>\n<start_of_turn>model\n",
]
NAMES = ["elephant", "spider", "eagle", "dolphin", "volcano", "desert", "library", "harbor",
         "joy", "fear", "jealousy", "serenity", "violin", "umbrella", "telescope", "candle"]
bank = load_bank("dev_bank.yaml")
result = collect_prg_hf(model, tok, bank, NAMES, layer=21, probe_layer=35,
                        alpha=1.0, prompts=PROMPTS, out="prg.jsonl")

In [ ]:
import numpy as np

from mirror.grading import RulesGrader, strip_prompt
from mirror.probes import prg, train_probe

npz = np.load("prg.jsonl.npz")
probe = train_probe(npz["activations"], npz["concept"], npz["prompt_id"])

grader = RulesGrader()
records = result["records"]
hits = 0
for r in records:
    g = grader.grade(r["concept"], strip_prompt(r["report"]))
    hits += int(g["identified"] in ("exact", "related"))
report_acc = hits / len(records)

print(f"probe accuracy (held-out prompts): {probe.accuracy:.3f}")
print(f"probe control (shuffled labels):   {probe.control_accuracy:.3f}")
print(f"verbal report accuracy:            {report_acc:.3f}")
print(f"PROBE-REPORT GAP:                  {prg(probe.accuracy, report_acc):.3f}")